# Run OpenAI AgentKit Through Portkey's AI Gateway

OpenAI's Swarm/AgentKit is a fantastic way to orchestrate multi-agent workflows. But in production, autonomous agents can rack up massive bills, hit rate limits, and hallucinate tool calls. 

In this cookbook, we'll route an OpenAI AgentKit workflow through Portkey's AI Gateway. You'll get deep observability into agent mindsets, set budget caps, and even route OpenAI agents to Anthropic's Claude Haiku 4.5—all with a 2-line integration.

## What You'll Learn
- Initializing AgentKit with Portkey's unified client
- Tracing multi-step agent interactions in a single visual waterfall
- Routing an OpenAI Agent to Claude Haiku 4.5

## Prerequisites
- Python 3.9+
- Portkey API Key ([get one free](https://app.portkey.ai))
- OpenAI API Key

**Time to complete**: ~5 minutes

*Note: Portkey currently processes over 500B tokens across 125M daily requests. Our native AgentKit tracing is trusted by thousands of organizations scaling autonomous systems.*

## Step 1: Installation

Install the necessary dependencies. We need `portkey-ai`, `openai`, and the Swarm/AgentKit package.

In [ ]:
!pip install -q portkey-ai openai git+https://github.com/openai/swarm.git

## Step 2: Initialize the Swarm Client with Portkey

The Swarm library uses the standard OpenAI client under the hood. To get gateway superpowers, we just modify the `base_url` and inject Portkey's headers during client creation.

In [ ]:
import os
from openai import OpenAI
from swarm import Swarm, Agent
from portkey_ai import PORTKEY_GATEWAY_URL, createHeaders

PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY", "your-portkey-api-key")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")

# Create an OpenAI client routed through Portkey
portkey_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=PORTKEY_GATEWAY_URL,
    default_headers=createHeaders(
        api_key=PORTKEY_API_KEY,
        provider="openai",
        # The trace_id groups all agent interactions into one session in the dashboard
        trace_id="agent_customer_support_001",
        metadata={
            "_environment": "production",
            "_user": "demo_user_123"
        }
    )
)

# Initialize Swarm with our upgraded client
swarm_client = Swarm(client=portkey_client)

print("✅ Swarm initialized with Portkey Gateway routing!")

## Step 3: Define Agents and Tools

Let's create a simple multi-agent setup: a Triage Agent that hands off to a Sales Agent or a Support Agent.

In [ ]:
def transfer_to_sales():
    return sales_agent

def transfer_to_support():
    return support_agent

def process_refund(item_id):
    print(f"[Tool Execution] Processing refund for {item_id}...")
    return f"Refund for {item_id} processed successfully."

triage_agent = Agent(
    name="Triage Agent",
    instructions="You are a receptionist. Direct users to either Sales or Support. Use the transfer functions.",
    functions=[transfer_to_sales, transfer_to_support],
    model="gpt-4o-mini"
)

sales_agent = Agent(
    name="Sales Agent",
    instructions="You are a highly enthusiastic sales agent.",
    model="gpt-4o-mini"
)

support_agent = Agent(
    name="Support Agent",
    instructions="You handle technical issues and refunds.",
    functions=[process_refund],
    model="gpt-4o"
)

## Step 4: Run the Multi-Agent Workflow

When we run this, Swarm will interact with the LLM multiple times. Because we supplied a `trace_id` when initializing the client, Portkey will group every single LLM call—and every single tool execution—into one beautiful observability trace.

In [ ]:
try:
    print("Starting agent interaction...\n")
    response = swarm_client.run(
        agent=triage_agent,
        messages=[{"role": "user", "content": "I bought a defective toaster (Item #883) and I want my money back!"}]
    )
    
    print(f"\n🤖 Final Agent: {response.agent.name}")
    print(f"💬 Response: {response.messages[-1]['content']}")
    print("\n✅ Check your Portkey dashboard! You will see the entire chain of thought, the handoff, and the tool call structured under 'agent_customer_support_001'.")

except Exception as e:
    print(f"❌ Error: {e}")

## 🔄 Switch Providers in One Line: Swarm with Claude!

Want to run Swarm using Anthropic's new, blazing fast Claude Haiku 4.5? You don't need to fork the Swarm repository. Portkey automatically translates OpenAI-formatted agent interactions into Anthropic-compatible payloads!

In [ ]:
try:
    # Initialize a new client targeting Anthropic
    anthropic_portkey_client = OpenAI(
        api_key=os.getenv("ANTHROPIC_API_KEY", "your-anthropic-key"),
        base_url=PORTKEY_GATEWAY_URL,
        default_headers=createHeaders(
            api_key=PORTKEY_API_KEY,
            provider="anthropic",  # ← Magic happens here
            trace_id="agent_claude_002"
        )
    )
    
    claude_swarm = Swarm(client=anthropic_portkey_client)
    
    # We must explicitly tell our agents to use an Anthropic model name
    haiku_agent = Agent(
        name="Claude Haiku Agent",
        instructions="You are a helpful assistant running on Anthropic infrastructure via Portkey.",
        model="claude-3-5-haiku-20241022"
    )
    
    print("\nStarting Anthropic agent interaction...")
    haiku_response = claude_swarm.run(
        agent=haiku_agent,
        messages=[{"role": "user", "content": "Briefly explain who you are."}]
    )
    
    print(f"\n💬 Claude says: {haiku_response.messages[-1]['content']}")
    
except Exception as e:
    print(f("❌ Make sure to set ANTHROPIC_API_KEY.\nError: {e}"))

## 🎯 Key Takeaways

1. **Agent Observability**: Autonomous agents easily burn through tokens. Tracing them grouped by `trace_id` ensures you see exactly what functions they execute and when.
2. **Framework Freedom**: Swarm is an OpenAI package, but Portkey's translation layer lets you run Swarm logic on Claude, Gemini, or Llama seamlessly.
3. **Zero Rewrites**: The integration requires changing exactly two lines of code (base URL and headers).

## Common Gotchas
- When routing to Anthropic through the standard OpenAI SDK, make sure the `Agent.model` parameter references a valid Anthropic model structure (e.g., `claude-3-5-sonnet-20241022`).

## 🚀 Next Steps

- **Add Budget Caps**: [Set cost limits on your agents so they don't bankrupt you](https://portkey.ai/docs/product/ai-gateway/virtual-keys)
- **Enable Guardrails**: [Block agents from executing unsafe tool calls](https://portkey.ai/docs/product/guardrails)
- **Analyze Traces**: Head to your [Portkey Dashboard](https://app.portkey.ai) to see the waterfall UI.

---

⭐ **Star our gateway**: [github.com/Portkey-AI/gateway](https://github.com/Portkey-AI/gateway)

📚 **Full docs**: [portkey.ai/docs](https://portkey.ai/docs/)

🐦 **Follow us**: [@PortkeyAI](https://x.com/PortkeyAI)
